# ETF Document Loader

Load the full ETF universe from FinanceDatabase into ChromaDB.
Data is sourced from the FinanceDatabase package (~36K ETFs, updated weekly).
No Tavily enrichment needed — the `summary` field from fund prospectuses is used directly.

In [1]:
%reload_ext autoreload
%autoreload 2

import sys
import os
from pathlib import Path

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../..')))

from apps.agentic.core.document_loaders.etf.finance_database_loader import FinanceDatabaseLoader
from apps.agentic.core.constants import ETF_DB_NAME, ETF_COLLECTION_NAME
from apps.agentic.core.utils import set_chatgpt_env

DB_PATH = Path("../../.db").resolve()
DB_PATH.mkdir(parents=True, exist_ok=True)

set_chatgpt_env(filedir="../../.keys")

## Configuration

In [2]:
ETF_DB_NAME, ETF_COLLECTION_NAME, DB_PATH

('etf', 'etf-info', PosixPath('/Users/troy/Develop/gly.fish/yada/.db'))

## Load ETFs

In [3]:
loader = FinanceDatabaseLoader(db_path=str(DB_PATH))
coll = loader.vectorstore._collection
print("Collection:", coll.name)
print("Docs before load:", coll.count())

Collection: etf-info
Docs before load: 0


In [4]:
# Test load — VanEck only (~386 funds, ~1 minute)
# For the full 36K universe: await loader.load_all_documents()
await loader.load_all_documents(family="VanEck Asset Management")
print("Docs after load:", coll.count())

INFO:     FinanceDatabaseLoader: downloading ETF data from FinanceDatabase...
INFO:     FinanceDatabaseLoader: 386 ETFs downloaded, building documents...
INFO:     FinanceDatabaseLoader: writing 386 documents to ChromaDB...
INFO:     FinanceDatabaseLoader: loaded 386 ETF documents into collection 'etf-info'.
Docs after load: 386


In [5]:
## Inspect stored documents
probe = coll.get(limit=5)
docs: list = probe.get("documents") or []
metas: list = probe.get("metadatas") or []
for i, (doc, meta) in enumerate(zip(docs, metas)):
    print(f"\n--- [{i}] {meta.get('ticker')} | {meta.get('family')} ---")
    print(doc[:400])


--- [0] 2TCB.BE | VanEck Asset Management ---
# VANECK VE.MU.-ASS.CON.AL.
- **Ticker:** 2TCB.BE
- **Fund Family:** VanEck Asset Management
- **Exchange:** BER
- **Currency:** EUR

--- [1] AFK | VanEck Asset Management ---
# VanEck Vectors Africa Index ETF
- **Ticker:** AFK
- **Fund Family:** VanEck Asset Management
- **Asset Class:** Equities
- **Category:** Frontier Markets
- **Exchange:** PCX
- **Currency:** USD

## Description
The investment seeks to replicate as closely as possible, before fees and expenses, the price and yield performance of the MVIS√Ç¬Æ GDP Africa Index.
 The fund normallyinvests at least 80%

--- [2] ANGL | VanEck Asset Management ---
# VanEck Vectors Fallen Angel High Yield Bond ETF
- **Ticker:** ANGL
- **Fund Family:** VanEck Asset Management
- **Asset Class:** Fixed Income
- **Category:** High Yield Bonds
- **Exchange:** NMS
- **Currency:** USD

## Description
The investment seeks to replicate as closely as possible, before fees and expenses, the price and y

In [6]:
## Similarity search — unfiltered
results = loader.vectorstore.similarity_search("international equity dividend growth", k=5)
for r in results:
    print(r.metadata.get("ticker"), "|", r.metadata.get("name"), "|", r.metadata.get("family"))
    print(r.page_content[:200])
    print()

TDIV.MI | VanEck Vectors Morningstar Developed Markets Dividend Leaders UCITS ETF | VanEck Asset Management
# VanEck Vectors Morningstar Developed Markets Dividend Leaders UCITS ETF
- **Ticker:** TDIV.MI
- **Fund Family:** VanEck Asset Management
- **Asset Class:** Information Technology
- **Category:** Dev

TDIV.L | VanEck Vectors Morningstar Developed Markets Dividend Leaders UCITS ETF | VanEck Asset Management
# VanEck Vectors Morningstar Developed Markets Dividend Leaders UCITS ETF
- **Ticker:** TDIV.L
- **Fund Family:** VanEck Asset Management
- **Asset Class:** Information Technology
- **Category:** Deve

TDIV.AS | VanEck Vectors Morningstar Developed Markets Dividend Leaders UCITS ETF | VanEck Asset Management
# VanEck Vectors Morningstar Developed Markets Dividend Leaders UCITS ETF
- **Ticker:** TDIV.AS
- **Fund Family:** VanEck Asset Management
- **Asset Class:** Information Technology
- **Category:** Dev

TDIV.SW | VanEck Vectors Morningstar Developed Markets Dividend Leaders

In [7]:
## Similarity search — filtered by family
results = loader.vectorstore.similarity_search(
    "bond income floating rate",
    k=5,
    filter={"family": "VanEck Asset Management"},
)
for r in results:
    print(r.metadata.get("ticker"), "|", r.metadata.get("name"))
    print(r.page_content[:200])
    print()

FLTR.MX | VanEck Vectors Investment Grade Floating Rate ETF
# VanEck Vectors Investment Grade Floating Rate ETF
- **Ticker:** FLTR.MX
- **Fund Family:** VanEck Asset Management
- **Asset Class:** Fixed Income
- **Category:** Corporate Bonds
- **Exchange:** MEX

FLTR | VanEck Vectors Investment Grade Floating Rate ETF
# VanEck Vectors Investment Grade Floating Rate ETF
- **Ticker:** FLTR
- **Fund Family:** VanEck Asset Management
- **Asset Class:** Fixed Income
- **Category:** Corporate Bonds
- **Exchange:** PCX
- 

FLOT.AX | VanEck Vectors Australian Floating Rate ETF
# VanEck Vectors Australian Floating Rate ETF
- **Ticker:** FLOT.AX
- **Fund Family:** VanEck Asset Management
- **Asset Class:** Fixed Income
- **Category:** Corporate Bonds
- **Exchange:** ASX
- **C

MBBB | VanEck Vectors Moody's Analytics BBB Corporate Bond ETF
# VanEck Vectors Moody's Analytics BBB Corporate Bond ETF
- **Ticker:** MBBB
- **Fund Family:** VanEck Asset Management
- **Asset Class:** Fixed Income
- **Cate